In [2]:
import torch
import torch.nn as nn
import math

In [38]:
x=torch.tensor([[1,2,3,4,5],[2,3,4,5,6]],dtype=torch.float32)
x,indice=torch.max(x,dim=1)
x

tensor([5., 6.])

In [16]:
class InputEmbeddings(nn.Module):
    def __init__(self,d_model,vocab_size):
        super().__init__()
        self.d_model=d_model
        self.vocab_size=voacb_size
        self.embedding=nn.Embedding(vocab_size,d_model)
    def forward(self,x):
        return self.embedding(x)*math.sqrt(self.d_model)

In [21]:
class PositionalEmbedding(nn.Module):
    def __init__(self,d_model,seq_len,dropout):
        super().__init__()
        self.d_model=d_model
        self.seq_len=seq_len
        self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(seq_len,d_model)
        # (seq_len,1)
        pos=torch.arange(0,seq_len).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(100000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div_term)
        pe[:,1::2]=torch.cos(pos*div_term)
        # (1,seq_len,d_model)
        pe=pe.unsqueeze(0)
        self.register_buffer('pe',pe)
    def forward(self,x):
        # 只切有限长度
        return self.dropout(x+self.pe[:,:x.shape[1],:])

In [1]:
class PositionalEmbedding(nn.Module):
    def __init__(self,d_model,seq_len,dropout):
        super().__init__()
        self.d_model=d_model
        self.seq_len=seq_len
        self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(seq_len,d_model)
        div_erm=torch.exp(torch.arange(0,d_model,2)*(torch.log(100000.0/d_model)))
        pos=torch.arange(0,seq_len).unsqueeze(1)
        pe[:,0::2]=torch.sin(pos*div_term)
        pe[:,1::2]=torch.cos(pos*div_term)
        pe=pe.unsqueeze(0)
        self.register_buffer('pe',pe)
    def forward(x):
        return self.dropout(x+self.pe[:,x.shape[1],:])
        

NameError: name 'nn' is not defined

In [7]:
class LayerNormalization(nn.Module):
    def __init__(self,eps:float=1e-6):
        self.eps=eps
        self.alpha=nn.Parameter(torch.ones(1))
        self.beta=nn.Parameter(torch.zeros(1))
    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        std=x.std(dim=-1,keepdim=True)
        return self.alpha*((x-mean)/(std+self.eps))+self.beta

In [8]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self,d_model:int,h:int,dropout:float):
        super().__init__()
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
        self.d_model=d_model
        self.h=h
        assert self.d_model%self.h==0
        self.d_k=self.d_model//self.h
        self.dropout=nn.Dropout(dropout)
    @staticmethod
    def attention(q,k,v,mask,dropout:nn.Dropout):
        d_k=q.shape[-1]
        attention_score=q@k.transpose(-2,-1)/math.sqrt(d_k)
        if mask is not None:
            attention_score=attention_score.masked_fill(mask==0,-1e9)
        attention_score=dropout(attention_score.softmax(dim=-1))
        return attention_score@v
    def forward(self,query,key,value,mask):
        q=self.w_q(query)
        k=self.w_k(key)
        v=self.w_v(value)
        q=q.view(q.shape[0],q.shape[1],self.h,self.d_k).transpose(1,2)
        k=k.view(k.shape[0],k.shape[1],self.h,self.d_k).transpose(1,2)
        v=v.view(v.shape[0],v.shape[1],self.h,self.d_k).transpose(1,2)
        x=MultiHeadAttentionBlock.attention(q,k,v,mask,self.dropout)
        x=x.transpose(1,2).contiguous().view(x.shape[0],-1,self.d_model)
        x=self.w_o(x)
        return x

In [9]:
class ResidualConnection(nn.Module):
    def __init__(self,dropout):
        super()._init__()
        self.dropout=dropout
        self.norm=LayerNormalization()
    def forward(self,x,sublayer):
        return x+self.dropout(sublayer(self.norm(x)))

In [20]:
class FeedForwardBlock(nn.Module):
    def __init__(self,d_model,d_ff:int,dropout:float):
        super().__init__()
        self.dropout=dropout
        self.d_model=d_model
        self.d_ff=d_ff
        self.linear1=nn.Linear(d_model,d_ff)
        self.linear2=nn.Linear(d_ff,d_model)
        self.dropout=nn.Dropout(dropout)
        self.relu=nn.Relu()
    def forward(self,x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [24]:
class CNN(nn.Module):
    def __init__(self,input_shape,output_shape):
        super().__init__()
        self.input_shape=input_shape
        self.output_shape=output_shape
        self.cnn1=nn.Sequential(nn.Conv2d(in_channels=input_shape,out_channels=16,kernel_size=5,padding=2,stride=1),
                               nn.BatchNorm2d(16),
                               nn.ReLU(),
                               nn.MaxPool2d(kernel_size=2,stride=2))
        self.cnn2=nn.Sequential(nn.Conv2d(in_channels=16,out_channels=32,kernel_size=5,padding=2,stride=1),
                                nn.BatchNorm2d(32),
                                nn.ReLU(),
                                nn.MaxPool2d(kernel_size=2,stride=2))

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,d_model,output_size):
        super().__init__()
        self.input_size=input_size
        self.d_model=d_model
        self.w_x=nn.Linear(input_size,d_model)
        self.w_h=nn.Linear(d_model,d_model)
        self.tanh=nn.Tanh()
        self.fc=nn.Linear(d_model,output_size)
    def forward(self,x):
        batch_size,seq_len,_=x.shape
        h=torch.zeros(batch_size,self.d_model)
        for i in range(seq_len):
            x_t=x[:,i,:]
            h=self.tanh(self.w_x(x)+self.w_h(h))
        return self.fc(h)

In [ ]:
class CNN(nn.Module):
    def __init__(self,input_channel):
        super().__init__()
        self.input_channel=input_channel
        self.cnn1=nn.Sequential(nn.Conv2d(in_channels=input_channel,out_channels=16,kernel_size=5,padding=2,stride=1),
                               nn.BatchNorm2d(16),
                               nn.ReLU(),
                               nn.MaxPool2d(kernel_size=2,stride=2))
        self.cnn2=nn.Sequential(nn.Conv2d(in_channels=16,out_channels=32,kernel_size=5,padding=2,stride=1),
                                nn.BatchNorm2d(32),
                                nn.ReLU(),
                                nn.MaxPool2d(kernel_size=2,stride=2))
    def forward(self,x):
        x=self.cnn1(x)
        x=self.cnn2(x)
        return x
        

In [1]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.w_x=nn.Linear(input_size,hidden_size)
        self.w_h=nn.Linear(hidden_size,hidden_size)
        self.input_size=input_size
        self.hidden_size=hidden_size
        self.fc=nn.Linear(hidden_size,output_size)
        self.tanh=nn.Tanh()
    def forward(self,x):
        batch_size,seq_len,input_size=x.shape
        h=torch.zeros(batch_size,self.hidden_size)
        for i in range(seq_len):
            xt=x[:,i,:]
            h=self.tanh(self.w_x(xt)+self.w_h(h))
        return self.fc(h)

NameError: name 'nn' is not defined

In [9]:
import torch
import torch.nn.functional as F
logits=torch.randn(5,3)
targets=torch.randint(0,3,(5,))
print(F.cross_entropy(logits,targets))

tensor(1.4141)


In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self,seq_len,d_model,dropout):
        super().__init__()
        self.seq_len=seq_len
        self.d_model=d_model
        pe=torch.zeros(seq_len,d_model)
        # (seq_len,1)
        pos=torch.arange(0,seq_len).unsqueeze(1)
        # (d_model//2,1)
        div_term=torch.exp(torch.arange(0,d_model,2).float()*(-torch.log(100000.0)/self.d_model))
        # 广播机制
        pe[:,0::2]=torch.sin(pos*div_term)
        pe[:,1::2]=torch.cos(pos*div_term)
        self.dropout=nn.Dropout(dropout)
        pe=pe.unsqueeze(0)
        self.register_buffer('pe',pe)
    def forward(self,x):
        return self.dropout(x+self.pe[:,:x.shape[1],:])
        

NameError: name 'nn' is not defined

In [3]:
 X = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
X.shape

torch.Size([5])

# Linear Regression
## Autograd

In [34]:
def function(x:torch.tensor):
    return 2*x+1

def linear_regression():
    # (5)
    x=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    # (5)
    y=function(x)
    # 权重
    # (1)
    w=torch.ones(1,requires_grad=True)
    # (1)
    b=torch.zeros(1,requires_grad=True)
    # 超参数
    epochs=1000
    lr=0.01
    for i in range(epochs):
        y_pred=w*x+b
        loss=torch.mean((y_pred-y)**2)
        loss.backward()
        with torch.no_grad():
            #必须原地操作
            w-=lr*w.grad
            b-=lr*b.grad
            w.grad.zero_()
            b.grad.zero_()
        if i % 100 == 0:
            print(f"Epoch {i}: Loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")

    return w, b
final_w, final_b =  linear_regression()

Epoch 0: Loss=18.0000, w=1.2800, b=0.0800
Epoch 100: Loss=0.0417, w=2.1321, b=0.5230
Epoch 200: Loss=0.0212, w=2.0942, b=0.6601
Epoch 300: Loss=0.0108, w=2.0671, b=0.7577
Epoch 400: Loss=0.0055, w=2.0478, b=0.8273
Epoch 500: Loss=0.0028, w=2.0341, b=0.8769
Epoch 600: Loss=0.0014, w=2.0243, b=0.9123
Epoch 700: Loss=0.0007, w=2.0173, b=0.9375
Epoch 800: Loss=0.0004, w=2.0123, b=0.9554
Epoch 900: Loss=0.0002, w=2.0088, b=0.9682


## Manual

In [12]:
def linear_regression():
    device="cuda" if torch.cuda.is_available() else "cpu"
    x=torch.tensor([1,2,3,4,5],dtype=torch.float32).to(device)
    y=torch.tensor([3,5,7,9,11],dtype=torch.float32).to(device)
    w=torch.tensor(0,device=device)
    b=torch.tensor(0,device=device)
    epochs=500
    lr=0.01
    for epoch in range(epochs):
        y_pred=w*x+b
        loss=torch.mean((y_pred-y)**2)
        # (5)
        error=y_pred-y
        # (1)
        # element-wise
        d_w=2*torch.mean(error*x)
        # (1)
        d_b=2*torch.mean(error)
        w=w-lr*d_w
        b=b-lr*d_b
        if epoch%100==0:
             print(f"Epoch {epoch}: Loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")
    return w,b
w,b=linear_regression()

Epoch 0: Loss=57.0000, w=2.5000, b=0.7000
Epoch 100: Loss=0.0010, w=2.0205, b=0.9260
Epoch 200: Loss=0.0000, w=2.0037, b=0.9865
Epoch 300: Loss=0.0000, w=2.0007, b=0.9976
Epoch 400: Loss=0.0000, w=2.0001, b=0.9996


## 显示解

In [32]:
def linear_regression():
    x=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    y=torch.tensor([3,5,7,9,11],dtype=torch.float32).unsqueeze(1)
    # (5,2)
    x=torch.stack([x,torch.ones_like(x)],dim=1)
    # (2,5)
    xt=x.t()
    theta=torch.inverse(xt@x)@xt@y
    print(f"w是{theta[0].item():.4f},b是{theta[1].item():.4f}")
    
linear_regression()

w是2.0000,b是1.0000


# Logistic Regression

## Autograd

In [26]:
import torch
def sigmoid(x:float):
    return 1.0/(1+torch.exp(-x))
def LogisticRegression():
    x=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    y=torch.tensor([1,0,1,0,0],dtype=torch.float32)
    w=torch.ones(1,requires_grad=True)
    b=torch.zeros(1,requires_grad=True)
    epochs=1000
    lr=0.01
    for i in range(epochs):
        z=sigmoid(w*x+b)
        # (1)
        loss=-torch.mean(y* torch.log(z)+(1-y)*torch.log(1-z))
        loss.backward()
        with torch.no_grad():
            w-=lr*w.grad
            b-=lr*b.grad
            w.grad.zero_()
            b.grad.zero_()
        if i % 100 == 0:
            print(f"Epoch {i}: Loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")
            
    return w, b

final_w, final_b = LogisticRegression()
        
        

Epoch 0: Loss=2.3027, w=0.9795, b=-0.0051
Epoch 100: Loss=0.6347, w=-0.1100, b=-0.2117
Epoch 200: Loss=0.6136, w=-0.2205, b=-0.1414
Epoch 300: Loss=0.6046, w=-0.2546, b=-0.0530
Epoch 400: Loss=0.5963, w=-0.2808, b=0.0340
Epoch 500: Loss=0.5887, w=-0.3054, b=0.1180
Epoch 600: Loss=0.5816, w=-0.3292, b=0.1988
Epoch 700: Loss=0.5750, w=-0.3521, b=0.2765
Epoch 800: Loss=0.5689, w=-0.3743, b=0.3514
Epoch 900: Loss=0.5633, w=-0.3957, b=0.4234


In [12]:
import torch
def function(x1,x2):
    return 3*x1+4*x2+5
def linear_regression():
    x1=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    x2=torch.tensor([3,2,1,-2,5],dtype=torch.float32)
    y=function(x1,x2)
    w1=torch.ones(1,dtype=torch.float32)
    w2=torch.ones(1,dtype=torch.float32)
    b=torch.ones(1,dtype=torch.float32)
    lr=1e-2
    for i in range(2000):
        y_pred=w1*x1+w2*x2+b
        loss=torch.mean((y_pred-y)**2)
        error=y_pred-y
        d_w1=torch.mean(2*error*x1)
        d_w2=torch.mean(2*error*x2)
        d_b=torch.mean(2*error)
        w1-=d_w1*lr
        w2-=d_w2*lr
        b-=d_b*lr
        print(f"w1={w1},w2={w2},b={b}\n")
    return w1,w2,b
linear_regression()

w1=tensor([2.0040]),w2=tensor([1.8760]),b=tensor([1.3080])

w1=tensor([2.6740]),w2=tensor([2.4818]),b=tensor([1.5181])

w1=tensor([3.1186]),w2=tensor([2.9035]),b=tensor([1.6619])

w1=tensor([3.4112]),w2=tensor([3.1995]),b=tensor([1.7610])

w1=tensor([3.6016]),w2=tensor([3.4093]),b=tensor([1.8300])

w1=tensor([3.7232]),w2=tensor([3.5601]),b=tensor([1.8785])

w1=tensor([3.7989]),w2=tensor([3.6700]),b=tensor([1.9134])

w1=tensor([3.8440]),w2=tensor([3.7516]),b=tensor([1.9391])

w1=tensor([3.8688]),w2=tensor([3.8134]),b=tensor([1.9586])

w1=tensor([3.8803]),w2=tensor([3.8611]),b=tensor([1.9740])

w1=tensor([3.8832]),w2=tensor([3.8989]),b=tensor([1.9867])

w1=tensor([3.8806]),w2=tensor([3.9294]),b=tensor([1.9976])

w1=tensor([3.8746]),w2=tensor([3.9545]),b=tensor([2.0074])

w1=tensor([3.8667]),w2=tensor([3.9756]),b=tensor([2.0164])

w1=tensor([3.8577]),w2=tensor([3.9936]),b=tensor([2.0250])

w1=tensor([3.8482]),w2=tensor([4.0092]),b=tensor([2.0332])

w1=tensor([3.8386]),w2=tensor([4.0228]),

(tensor([3.0016]), tensor([4.0004]), tensor([4.9934]))

In [18]:
import torch
def function(x1,x2):
    return 3*x1+4*x2+5
def linear_regression():
    x1=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    x2=torch.tensor([3,2,1,-2,5],dtype=torch.float32)
    y=function(x1,x2)
    w1=torch.ones(1,dtype=torch.float32,requires_grad=True)
    w2=torch.ones(1,dtype=torch.float32,requires_grad=True)
    b=torch.ones(1,dtype=torch.float32,requires_grad=True)
    lr=1e-2
    for i in range(2000):
        y_pred=w1*x1+w2*x2+b
        loss=torch.mean((y_pred-y)**2)
        loss.backward()
        with torch.no_grad():
            w1-=lr*w1.grad
            w2-=lr*w2.grad
            b-=lr*b.grad
            w1.grad.zero_()
            w2.grad.zero_()
            b.grad.zero_()
        print(f"w1={w1},w2={w2},b={b}\n")
    return w1,w2,b
linear_regression()

w1=tensor([2.0040], requires_grad=True),w2=tensor([1.8760], requires_grad=True),b=tensor([1.3080], requires_grad=True)

w1=tensor([2.6740], requires_grad=True),w2=tensor([2.4818], requires_grad=True),b=tensor([1.5181], requires_grad=True)

w1=tensor([3.1186], requires_grad=True),w2=tensor([2.9035], requires_grad=True),b=tensor([1.6619], requires_grad=True)

w1=tensor([3.4112], requires_grad=True),w2=tensor([3.1995], requires_grad=True),b=tensor([1.7610], requires_grad=True)

w1=tensor([3.6016], requires_grad=True),w2=tensor([3.4093], requires_grad=True),b=tensor([1.8300], requires_grad=True)

w1=tensor([3.7232], requires_grad=True),w2=tensor([3.5601], requires_grad=True),b=tensor([1.8785], requires_grad=True)

w1=tensor([3.7989], requires_grad=True),w2=tensor([3.6700], requires_grad=True),b=tensor([1.9134], requires_grad=True)

w1=tensor([3.8440], requires_grad=True),w2=tensor([3.7516], requires_grad=True),b=tensor([1.9391], requires_grad=True)

w1=tensor([3.8688], requires_grad=True),

(tensor([3.0016], requires_grad=True),
 tensor([4.0004], requires_grad=True),
 tensor([4.9934], requires_grad=True))

## Manual

In [27]:
import torch
def sigmoid(x:float):
    return 1.0/(1+torch.exp(-x))
def LogisticRegression():
    x=torch.tensor([1,2,3,4,5],dtype=torch.float32)
    y=torch.tensor([1,0,1,0,0],dtype=torch.float32)
    w=torch.ones(1)
    b=torch.zeros(1)
    epochs=1000
    lr=0.01
    for i in range(epochs):
        z=sigmoid(w*x+b)
        loss=-torch.mean(y* torch.log(z)+(1-y)*torch.log(1-z))
        d_w=torch.mean((z-y)*x)
        d_b=torch.mean(z-y)
        w-=lr*d_w
        b-=lr*d_b
        if i % 100 == 0:
            print(f"Epoch {i}: Loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")
            
    return w, b

final_w, final_b = LogisticRegression()

Epoch 0: Loss=2.3027, w=0.9795, b=-0.0051
Epoch 100: Loss=0.6347, w=-0.1100, b=-0.2117
Epoch 200: Loss=0.6136, w=-0.2205, b=-0.1414
Epoch 300: Loss=0.6046, w=-0.2546, b=-0.0530
Epoch 400: Loss=0.5963, w=-0.2808, b=0.0340
Epoch 500: Loss=0.5887, w=-0.3054, b=0.1180
Epoch 600: Loss=0.5816, w=-0.3292, b=0.1988
Epoch 700: Loss=0.5750, w=-0.3521, b=0.2765
Epoch 800: Loss=0.5689, w=-0.3743, b=0.3514
Epoch 900: Loss=0.5633, w=-0.3957, b=0.4234


# Softmax Cross-Entropy

## Autograd

In [19]:
import torch
import torch.nn.functional as F

def autograd_multiclass():
    # 1. 模拟数据: 5个样本, 2个特征, 3个类别
    X = torch.randn(5, 2)
    y = torch.tensor([0, 1, 2, 0, 1]) # 真实类别索引
    
    # 2. 初始化参数 (D=2, C=3)
    W = torch.randn(2, 3, requires_grad=True)
    b = torch.randn(3, requires_grad=True)
    
    lr = 1e-1
    for i in range(500):
        # 前向传播
        logits = X @ W + b
        
        # PyTorch 的 cross_entropy 内部自带了 Softmax
        loss = F.cross_entropy(logits, y)
        
        # 反向传播
        loss.backward()
        
        # 更新参数
        with torch.no_grad():
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_()
            b.grad.zero_()
            
        if i % 100 == 0:
            print(f"Iter {i}, Loss: {loss.item():.4f}")
            
    return W, b

autograd_multiclass()

Iter 0, Loss: 3.5735
Iter 100, Loss: 0.4185
Iter 200, Loss: 0.3448
Iter 300, Loss: 0.3130
Iter 400, Loss: 0.2951


(tensor([[-2.4084,  0.9723,  2.6501],
         [ 1.8795, -1.5332,  0.7789]], requires_grad=True),
 tensor([ 0.5353, -0.7580,  1.3450], requires_grad=True))

## Manual

In [15]:
import torch
def cross_entropy(logits,targets):
    # (n,l)
    n=logits.shape[0]
    logits=logits-torch.max(logits,dim=-1,keepdim=True)[0]
    probs=torch.exp(logits)/torch.sum(torch.exp(logits),dim=-1,keepdim=True)
    pred=probs[range(n),targets]
    loss=-torch.mean(torch.log(pred))
    return loss
x=torch.tensor([[0.2,0.8],[0.3,0.7]])
y=torch.tensor([1,0])
cross_entropy(x,y)

tensor(0.6753)

In [1]:
def cross_entropy(logits,targets):
    n= logits.shape[0]
    logits=logits-torch.max(logits,dim=-1,keepdim=True)[0]
    probs=torch.exp(logits)/torch.sum(torch.exp(logits),dim=-1,keepdim=True)
    pred=probs[range(n),targets]
    loss=-torch.mean(torch.log(pred))
    return loss

In [4]:
def cross_entropy(logits,targets):
    # (n,d)
    logits=logits-torch.max(logits,keepdim=True,dim=-1)[0]
    # (n,d)
    probs=torch.exp(logits)/torch.sum(torch.exp(logits),keepdim=True,dim=-1)
    preds=probs[:,targets]
    return -torch.mean(torch.log(preds))

In [20]:
import torch
import torch.nn.functional as F
def cross_entropy(logits,targets):
    # (n,l)
    n=logits.shape[0]
    logits=logits-torch.max(logits,dim=-1,keepdim=True)[0]
    # (n,l)
    probs=torch.exp(logits)/torch.sum(torch.exp(logits),dim=-1,keepdim=True)
    # (l)
    pred=probs[range(n),targets]
    
    loss=-torch.mean(torch.log(pred))
    return probs,loss

    
def multiclass():
    # 1. 模拟数据: 5个样本, 2个特征, 3个类别
    X = torch.randn(5, 2)
    n=X.shape[0]
    y = torch.tensor([0, 1, 2, 0, 1]) # 真实类别索引
    # 2. 初始化参数 (D=2, C=3)
    W = torch.randn(2, 3)
    b = torch.randn(3)
    
    lr = 1e-1
    for i in range(500):
        # 前向传播
        # (5,3)
        logits = X @ W + b
        # (5,3)
        probs,loss=cross_entropy(logits,y)
        probs[range(n),y]-=1
        # (2,3)
        d_w=(X.t()@(probs))
        # (3)
        d_b = torch.sum(probs,dim=0)
        W-=lr*d_w
        b-=lr*d_b
            
        if i % 100 == 0:
            print(f"Iter {i}, Loss: {loss.item():.4f}")
            
    return W, b

multiclass()

Iter 0, Loss: 2.0927
Iter 100, Loss: 0.4087
Iter 200, Loss: 0.3199
Iter 300, Loss: 0.2660
Iter 400, Loss: 0.2277


(tensor([[-3.2145,  2.5241,  3.5684],
         [ 6.0948, -0.3479, -4.9962]]),
 tensor([ 5.8051,  2.3515, -3.3749]))

# K-means

## Manual

In [35]:
import numpy as np

class KMeansScratch:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters

    def fit(self, X):
        # 1. 随机初始化质心 (从原始数据中随机选 k 个点)
        # (k)
        idx = np.random.choice(len(X), self.k, replace=False)
        # (k,d)
        self.centroids = X[idx]

        for _ in range(self.max_iters):
            # 2. 分配阶段：计算每个点到所有质心的距离，归类到最近的那一个
            # 距离公式: sqrt(sum((a-b)^2))
            # （N,K,D）-> (N,K)
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            # (N,1)
            labels = np.argmin(distances, axis=1)

            # 3. 更新阶段：计算每个簇内所有点的平均值，作为新的质心
            # (k,2)
            new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(self.k)])

            # 如果质心不再变化，提前退出
            if np.all(self.centroids == new_centroids):
                break
            self.centroids = new_centroids
            
        self.labels = labels
        return self

# 测试数据
# (n,d)
X = np.random.rand(100, 2)
model = KMeansScratch(k=3).fit(X)
print(f"手写版质心位置:\n{model.centroids}")

手写版质心位置:
[[0.77311452 0.26866766]
 [0.25314196 0.38316097]
 [0.61061062 0.79388868]]


In [10]:
import numpy as np
class KMeansScratch:
    def __init__(self, k=3, iterations=100):
        self.k=k
        self.iterations=iterations
    def fit(self,X):
        idx=np.random.choice(X.shape[0],self.k)
        self.centroids=X[idx]
        for i in range(self.iterations):
            distances=np.linalg.norm(X[:,np.newaxis]-self.centroids,axis=2)
            labels=np.argmin(distances,axis=1)
            new_centroids=np.array([X[labels==i].mean(axis=0) for i in range(self.k)])
            if np.all(self.centroids==new_centroids):
                break
            self.centroids=new_centroids
        self.labels=labels
        return self
X = np.random.rand(100, 2)
model = KMeansScratch(k=3).fit(X)
print(f"手写版质心位置:\n{model.centroids}")

手写版质心位置:
[[0.32245056 0.1961736 ]
 [0.30699695 0.82352896]
 [0.76294906 0.48320705]]


In [3]:
import numpy as np
class KMeansScratch:
    def __init__(self,k=3,iterations=100):
        self.k=k
        self.iterations=iterations
    def fit(self,X):
        # (n)->(k)
        idx=np.random.choice(X.shape[0],self.k)
        # (k,d)
        self.centroids=X[idx]
        for i in range(self.iterations):
            # (n,k,d)->(n,k)
            distances=np.linalg.norm(X[:,np.newaxis,:]-self.centroids,axis=2)
            # (n,k)->(n)
            labels=np.argmin(distances,axis=1)
            # (k,d)
            new_centroids=np.array([X[labels==i].mean(axis=0) for i in range(self.k)])
            if(np.all(new_centroids==self.centroids)):
                break;
            self.centroids=new_centroids
            return self
X=np.random.rand(100,2)
model=KMeansScratch().fit(X)
print(f"手写版质心位置:\n{model.centroids}")
            
        

手写版质心位置:
[[0.4765947  0.21664838]
 [0.02817825 0.82454867]
 [0.50805939 0.7588316 ]]


## Numpy

In [2]:
from sklearn.cluster import KMeans
import numpy as np

# 生成数据
X = np.random.rand(100, 2)

# 调用官方类
# n_init='auto' 会自动选择最佳的初始化次数
kmeans = KMeans(n_clusters=3, n_init='auto', random_state=42)
kmeans.fit(X)

print(f"官方版质心位置:\n{kmeans.cluster_centers_}")
print(f"前5个样本的标签: {kmeans.labels_[:5]}")

官方版质心位置:
[[0.21373007 0.49611013]
 [0.69034205 0.77904821]
 [0.60713591 0.19879557]]
前5个样本的标签: [1 2 2 1 2]


In [1]:
from sklearn.cluster import KMeans
import numpy as np
X=np.random.rand(100,2)
kmeans=KMeans(n_clusters=3,random_state=42)
kmeans.fit(X)
print(kmeans.cluster_centers_)

[[0.50386465 0.80126469]
 [0.2327301  0.26845718]
 [0.80047946 0.37728292]]


# MultihaedAttention

In [38]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,h,dropout):
        super().__init__()
        self.d_model=d_model
        self.h=h
        self.dropout=nn.Dropout(dropout)
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
        assert d_model%h==0
        self.d_h=d_model//h
    @staticmethod
    def attention(query,key,value,mask,dropout):
        d_k=query.shape[-1]
        attention_score=query@key.transpose(-2,-1)/math.sqrt(d_k)
        if mask is not None:
            attention_score=attention_score.masked_fill(mask==0,-1e6)
        attention_score=dropout(attention_score.softmax(dim=-1))
        return attention_score@value
        
    def forward(self,query,key,value,mask):
        q=self.w_q(query).view(query.shape[0],query.shape[1],self.h,self.d_h).transpose(1,2) 
        k=self.w_k(key).view(query.shape[0],query.shape[1],self.h,self.d_h).transpose(1,2) 
        v=self.w_v(value).view(query.shape[0],query.shape[1],self.h,self.d_h).transpose(1,2) 
        x=MultiHeadAttention.attention(q,k,v,mask,self.dropout)
        x=x.transpose(1,2).contiguous().view(x.shape[0],-1,self.d_model)
        x=self.w_o(x)
        return x

In [ ]:
class CNN(nn.Module):
    def __init__(self,input_size,output_size):
        self.cnn1=nn.Sequential(nn.Conv2d(in_channels=input_size,out_channels=16,kernel_size=5,padding=1,stride=2),
                               nn.BatchNorm2d(16),
                               nn.Relu(),
                               nn.MaxPool2d(kernel_size=2,stride=2))
        self.cnn2=nn.Sequential(nn.Conv2d(in_channels=16,out_channels=output_size,kernel_size=5,padding=1,stride=2),
                               nn.BatchNorm2d(16),
                               nn.Relu(),
                               nn.MaxPool2d(kernel_size=2,stride=2))
        self.fc=nn.Linear()

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size):
        self.hidden_size=hidden_size
        self.input_size=input_size
        self.w_c=self.Linear(input_size,hidden_size)
        self.w_h=self.Linear(hidden_size,hidden_size)
        self.tanh=self.Tanh()
    def forward(self,x):
        batch_size,seq_len,input_size=x
        h=torch.zeros(batch_size,self.hidden_size)
        for i in range(seq_len):
            xt=x[:,i,:]
            h=self.w_c(xt)+self.w_h(h)
        return h

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,h,dropout):
        self.d_model=d_model
        self.h=h
        self.dropout=nn.Dropout(dropout)
        assert self.d_model % self.h==0
        self.d_h=self.d_model//self.h
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
    @staticmethod
    def attention(q,k,v,mask):
        d_model=q.shape[-1]
        attention_scores=q@k.transpose(-2,-1)/math.sqrt(d_model)
        if mask is not None:
            attention_scores=attention_scores.masked_fill(mask==0,-1e6)
        attention_score=self.dropout(attention_scores.softmax(dim=-1))
        return attention_score@v
    def forward(self,query,key,value,mask):
        q=self.w_q(query).view(q.shape[0],q.shape[1],self.h,self.d_h).transpose(1,2)
        k=self.w_k(key).view(k.shape[0],k.shape[1],self.h,self.d_h).transpose(1,2)
        v=self.w_v(value).view(v.shape[0],v.shape[1],self.h,self.d_h).transpose(1,2)
        o=MultiHeadAttention(q,k,v,mask)
        o.transpose(1,2).contiguous().view(q.shape[0],-1,self.d_model)
        return self.w_o(o)

NameError: name 'nn' is not defined

In [12]:
class FeedForwardNetwork(nn.Module):
    def __init__(self,d_model,d_ff,dropout):
        self.d_model=d_model
        self.d_ff=d_ff
        self.dropout=nn.Dropout(dropout)
        self.relu=nn.Relu()
        self.hidden1=nn.Linear(d_model,d_ff)
        self.hidden2=nn.Linear(d_ff,d_model)
    def forward(self,x):
        return self.hidden2(self.dropout(self.relu(self.hidden1)))
        

NameError: name 'nn' is not defined

In [17]:
class ResidualNetwork(nn.Module):
    def __init__(self,dropout):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        self.norm=LayerNormalization()
    def forward(self,x,sublayer):
        return x+self.dropout(sublayer(self.norm(x)))

NameError: name 'nn' is not defined

In [ ]:
class CNN(nn.Module):
    def __init__(self,input_size,output_size):
        self.input_size=input_size
        self.output_size=output_size
        self.cnn1=nn.Sequential(nn.Conv2d(in_channels=input_size,out_channels=output_size,kernel_size=5,padding=1,stride=2),
                               nn.BatchNorm2d(output_size),
                               nn.Relu(),
                               nn.MaxPool2d(kernel_size=2,stride=2))
        self.cnn2=nn.

        

In [1]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size):
        super().__init__()
        self.input_size=input_size
        self.hidden_size=hidden_size
        self.w_i=nn.Linear(input_size,hidden_size)
        self.w_h=nn.Linear(hidden_size,hidden_size)
        self.tanh=nn.Tanh()
    def forward(self,x):
        batch_size,seq_len,_=x.shape
        h=nn.zeros(batch_size,self.hidden_size)
        for i in range(seq_len):
            x_t=x[:,i,:]
            h=self.w_i(x_t)+self.w_h(h)
            h=self.tanh(h)
        return h

NameError: name 'nn' is not defined

In [1]:
import torch
def function(x):
    return 4*x+2
def linear_regression():
    x=torch.tensor([1,2,3],dtype=torch.float32)
    y=function(x)
    w=torch.tensor(0,dtype=torch.float32)
    b=torch.tensor(0,dtype=torch.float32)
    lr=1e-2
    for i in range(1000):
        pred=w*x+b
        loss=torch.mean((pred-y)**2)
        dw=2*torch.mean((pred-y)*x)
        db=2*torch.mean(pred-y)
        w-=dw*lr
        b-=db*lr
    print(f"w={w},b={b}")
linear_regression()

w=4.007983684539795,b=1.98185133934021


In [2]:
def linear_regression():
    x=torch.tensor([1,2,3],dtype=torch.float32)
    y=function(x)
    w=torch.tensor(0,dtype=torch.float32,requires_grad=True)
    b=torch.tensor(0,dtype=torch.float32,requires_grad=True)
    lr=1e-2
    for i in range(1000):
        error=x*w+b
        loss=torch.mean((error-y)**2)
        loss.backward()
        with torch.no_grad():
            w-=lr*w.grad
            b-=lr*b.grad
            w.grad.zero_()
            b.grad.zero_()
    print(f"w={w},b={b}")
linear_regression()

w=4.007983684539795,b=1.98185133934021


In [22]:
def sigmoid(x):
    return 1.0/(1.0+torch.exp(-x))
def logistic_regression():
    x=torch.tensor([1,2,3],dtype=torch.float32)
    y=torch.tensor([0,1,0],dtype=torch.float32)
    w=torch.tensor(0,dtype=torch.float32)
    b=torch.tensor(0,dtype=torch.float32)
    lr=1e-1
    for i in range(1000):
        pred=sigmoid(w*x+b)
        loss=-torch.mean(y*torch.log(pred)+(1-y)*torch.log(1-pred))
        dw=torch.mean((pred-y)*x)
        db=torch.mean(pred-y)
        w-=dw*lr
        b-=db*lr
        print(f"loss={loss}")
    print(f"w={w},b={b}")
logistic_regression()
        

loss=0.6931471824645996
loss=0.680218517780304
loss=0.6706127524375916
loss=0.6634616255760193
loss=0.6581222414970398
loss=0.6541222333908081
loss=0.6511145234107971
loss=0.6488444805145264
loss=0.6471243500709534
loss=0.6458160281181335
loss=0.6448166966438293
loss=0.6440503001213074
loss=0.6434597373008728
loss=0.6430025100708008
loss=0.6426464915275574
loss=0.6423675417900085
loss=0.6421473026275635
loss=0.6419720649719238
loss=0.6418311595916748
loss=0.641716718673706
loss=0.6416225433349609
loss=0.6415439248085022
loss=0.6414773464202881
loss=0.6414199471473694
loss=0.6413698792457581
loss=0.6413252949714661
loss=0.6412850618362427
loss=0.6412481069564819
loss=0.6412138938903809
loss=0.6411817073822021
loss=0.6411510109901428
loss=0.6411218047142029
loss=0.6410935521125793
loss=0.6410660743713379
loss=0.641039252281189
loss=0.6410130262374878
loss=0.6409872174263
loss=0.6409618258476257
loss=0.6409367918968201
loss=0.640911877155304
loss=0.6408872604370117
loss=0.6408630013465881

In [24]:
def cross_entropy(logits,target):
    logits=logits-torch.max(logits,keepdim=True,dim=-1)[0]
    preds=torch.exp(logits)/torch.sum(torch.exp(logits),keepdim=True,dim=-1)
    n=logits.shape[0]
    labels=preds[range(n),target]
    loss=-torch.mean(torch.log(labels))
    return preds,loss

In [28]:
def pp():
    x=torch.randn(3,2,dtype=torch.float32)
    y=torch.tensor([0,1,0],dtype=torch.float32)
    w=torch.randn(2,5,dtype=torch.float32)
    b=torch.tensor(5,dtype=torch.float32)
    lr=1e-1
    for i in range(1000):
        pred=x@w+b
        preds,loss=cross_entropy(pred,y)
        n=preds.shape[0]
        preds[range(n),y]-=1
        dw=x.t()@preds
        db=torch.sum(preds,dim=0)
        w-=dw*lr
        b-=db*lr
        print(f"loss={loss}")
    print(f"w={w},b={b}")
pp()

IndexError: tensors used as indices must be long, int, byte or bool tensors